In [1]:
import os
import json
from pathlib import Path

In [3]:
# Remove array from string
file = 'qwen3-8b-0-constrained-answers.jsonl'
with open(f"results/{file.replace('-answers.jsonl','-1-answers.jsonl')}", 'w') as f2:
    with open(f'results/{file}', 'r', encoding='utf-8') as f:
        for line in f:
            l = json.loads(line.strip())
            l['solution'] = l['solution'][0] if type(l['solution'])==list else l['solution']
            f2.write(json.dumps(l) + '\n')

In [14]:
def process_sanitized_files(folder_path: str) -> None:
    folder = Path(folder_path)

    for file in folder.iterdir():
        if not file.name.endswith("-sanitized.jsonl"):
            continue

        # Read all records
        records = []
        with open(file, "r") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))

        if not records:
            print(f"Skipping {file.name} — no records found.")
            continue

        # Sort by the numeric suffix after '/' e.g. "BigCodeBench/4" → 4
        def task_sort_key(r):
            tid = str(r["task_id"])
            part = tid.rsplit("/", 1)[-1]
            return int(part) if part.isdigit() else part

        records.sort(key=task_sort_key)

        # Split into two halves
        mid = len(records) // 2
        halves = [records[:mid], records[mid:]]

        # Write output files
        stem = file.stem  # e.g. "data-sanitize"
        for i, half in enumerate(halves, start=1):
            out_path = Path('resultsb-sanitized') / f"{stem}_part{i}.jsonl"
            with open(out_path, "w") as f:
                for record in half:
                    f.write(json.dumps(record) + "\n")
            print(f"Written {len(half)} records → {out_path.name}")

In [15]:
process_sanitized_files('resultsb')

Written 570 records → gemma2:9b-1-no-constrained-1-answers-sanitized_part1.jsonl
Written 570 records → gemma2:9b-1-no-constrained-1-answers-sanitized_part2.jsonl
Written 570 records → qwen3:8b-0-no-constrained-1-answers-sanitized_part1.jsonl
Written 570 records → qwen3:8b-0-no-constrained-1-answers-sanitized_part2.jsonl
Written 570 records → gemma2:2b-0-no-constrained-1-answers-sanitized_part1.jsonl
Written 570 records → gemma2:2b-0-no-constrained-1-answers-sanitized_part2.jsonl
Written 570 records → gemma2:2b-4-no-constrained-1-answers-sanitized_part1.jsonl
Written 570 records → gemma2:2b-4-no-constrained-1-answers-sanitized_part2.jsonl
Written 570 records → qwen3:8b-4-no-constrained-1-answers-sanitized_part1.jsonl
Written 570 records → qwen3:8b-4-no-constrained-1-answers-sanitized_part2.jsonl
Written 570 records → gemma2:2b-3-no-constrained-1-answers-sanitized_part1.jsonl
Written 570 records → gemma2:2b-3-no-constrained-1-answers-sanitized_part2.jsonl
Written 570 records → qwen3:8b-3